# Misinformation at Scale: Linguistic & Community Analysis

## Objective

This notebook performs deeper exploratory analysis on the cleaned Reddit dataset, focusing on:

1. **Linguistic patterns** — vocabulary, readability, linguistic markers
2. **Language characteristics** — sentiment proxies, tone analysis
3. **Thematic exploration** — frequent terms, n-grams by class
4. **Author behavior** — participation patterns, comment distribution
5. **Community structure** — interaction patterns, temporal dynamics
6. **Correlation analysis** — features associated with each class

The goal is to identify **statistically significant linguistic features** that may discriminate between misinformation and control discourse.

---

## 1. Setup & Load Data

In [3]:
import os
import sys
from pathlib import Path

# Add src to path using current directory
project_root = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(project_root, 'src'))

# Standard libraries
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import string
import logging
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.size'] = 10

# Download NLTK data if needed
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)

print("✓ Environment initialized")

✓ Environment initialized


In [4]:
# Load processed data from Notebook 01
processed_path = os.path.join(project_root, "data", "processed", "reddit_comments_for_modeling.csv")

try:
    df = pd.read_csv(processed_path)
    print(f"✓ Loaded {len(df):,} records from {processed_path}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
except FileNotFoundError:
    print(f"⚠ File not found: {processed_path}")
    print("Using raw data instead...")
    raw_path = os.path.join(project_root, "data", "raw", "reddit_comments_2020.csv")
    df = pd.read_csv(raw_path)
    print(f"✓ Loaded {len(df):,} records from raw data")

⚠ File not found: c:\Users\sanja\OneDrive\Documents\GitHub\misinformation-at-scale\data\processed\reddit_comments_for_modeling.csv
Using raw data instead...
✓ Loaded 10,000 records from raw data


## 2. Linguistic Feature Engineering

In [7]:
# Add linguistic features to the dataset
import re

# First, add labels if not present
if 'label' not in df.columns:
    misinformation_subs = ['conspiracy', 'NoNewNormal', 'theDonald']
    control_subs = ['askscience', 'science', 'news']
    df['label'] = df['subreddit'].apply(
        lambda x: 1 if x in misinformation_subs else (0 if x in control_subs else -1)
    )
    df = df[df['label'] != -1].copy()

def add_linguistic_features(df):
    """Add linguistic features to dataframe"""
    df = df.copy()
    
    # Word count
    df['word_count'] = df['body'].str.split().apply(len)
    
    # Sentence count
    df['sentence_count'] = df['body'].str.count(r'[.!?]+') + 1
    
    # Average word length
    df['avg_word_length'] = df['body'].str.split().apply(
        lambda words: np.mean([len(w) for w in words]) if words else 0
    )
    
    # Capitalization ratio
    df['capitalization_ratio'] = df['body'].apply(
        lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
    )
    
    # Digit count
    df['digit_count'] = df['body'].str.count(r'\d')
    
    # Question marks
    df['question_marks'] = df['body'].str.count(r'\?')
    
    # Exclamation marks
    df['exclamation_marks'] = df['body'].str.count(r'\!')
    
    # URL count
    df['url_count'] = df['body'].str.count(r'http|www')
    
    return df

df = add_linguistic_features(df)
print(f"✓ Added linguistic features to {len(df):,} records")
print(f"\nFeature columns: {[c for c in df.columns if c not in ['author', 'body', 'created_utc', 'score', 'subreddit', 'id', 'label']]}")

✓ Added linguistic features to 10,000 records

Feature columns: ['word_count', 'sentence_count', 'avg_word_length', 'capitalization_ratio', 'digit_count', 'question_marks', 'exclamation_marks', 'url_count']


## 3. Linguistic Feature Comparison

In [8]:
# Compare linguistic features by class
print("\n" + "="*70)
print("LINGUISTIC FEATURES BY CLASS")
print("="*70)

feature_cols = [
    "word_count",
    "sentence_count",
    "avg_word_length",
    "capitalization_ratio",
    "digit_count",
    "question_marks",
    "exclamation_marks",
    "url_count"
]

stats = df.groupby('label')[feature_cols].mean().reset_index()
stats['label_name'] = stats['label'].map({0: 'Control', 1: 'Misinformation'})

# Display comparison
print(f"\n{'Feature':<25} {'Control':<15} {'Misinformation':<15} {'Difference':<15}")
print("-" * 70)
for feature in feature_cols:
    control_avg = stats[stats['label'] == 0][feature].values[0]
    misinfo_avg = stats[stats['label'] == 1][feature].values[0]
    diff = misinfo_avg - control_avg
    pct_change = (diff / control_avg * 100) if control_avg != 0 else 0
    
    print(f"{feature:<25} {control_avg:<15.2f} {misinfo_avg:<15.2f} {pct_change:>+6.1f}%")


LINGUISTIC FEATURES BY CLASS

Feature                   Control         Misinformation  Difference     
----------------------------------------------------------------------
word_count                8.94            8.58              -4.1%
sentence_count            1.00            1.50             +49.8%
avg_word_length           5.66            5.11              -9.8%
capitalization_ratio      0.04            0.40            +971.8%
digit_count               0.70            0.33             -52.6%
question_marks            0.00            0.10              +0.0%
exclamation_marks         0.00            1.19              +0.0%
url_count                 0.00            0.00              +0.0%


In [ ]:
# Visualize key linguistic features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_to_plot = [
    ('capitalization_ratio', 'Capitalization Ratio', 0, 0),
    ('exclamation_marks', 'Exclamation Marks', 0, 1),
    ('question_marks', 'Question Marks', 1, 0),
    ('sentence_count', 'Sentence Count', 1, 1)
]

for feature, title, row, col in features_to_plot:
    ax = axes[row, col]
    
    for label, color, name in [(0, '#2ecc71', 'Control'), (1, '#e74c3c', 'Misinformation')]:
        data = df[df['label'] == label][feature]
        ax.hist(data, bins=30, alpha=0.6, label=name, color=color)
    
    ax.set_xlabel(title, fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{title} Distribution by Class', fontsize=12, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Vocabulary Analysis

In [ ]:
# Convert to Pandas for vocabulary analysis
df_pd = df_ling.select('body', 'label').toPandas()

# Initialize stopwords
stop_words = set(stopwords.words('english'))

def extract_terms(text, remove_stopwords=True, min_length=3):
    """Extract meaningful terms from text."""
    if not isinstance(text, str):
        return []
    
    # Lowercase and tokenize
    tokens = text.lower().split()
    
    # Remove punctuation and clean
    tokens = [t.strip(string.punctuation) for t in tokens]
    tokens = [t for t in tokens if t and len(t) >= min_length]
    
    # Remove stopwords
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    
    return tokens

# Extract terms by class
control_terms = []
misinfo_terms = []

for idx, row in df_pd.iterrows():
    terms = extract_terms(row['body'])
    if row['label'] == 0:
        control_terms.extend(terms)
    else:
        misinfo_terms.extend(terms)

print(f"\n✓ Extracted {len(control_terms):,} control terms")
print(f"✓ Extracted {len(misinfo_terms):,} misinformation terms")

In [ ]:
# Analyze most common terms
print("\n" + "="*70)
print("TOP TERMS BY CLASS")
print("="*70)

control_freq = Counter(control_terms)
misinfo_freq = Counter(misinfo_terms)

top_n = 15

print(f"\nControl (Science) - Top {top_n} terms:")
for term, count in control_freq.most_common(top_n):
    pct = (count / len(control_terms)) * 100
    print(f"  {term:20} | {count:6,} ({pct:5.2f}%)")

print(f"\nMisinformation - Top {top_n} terms:")
for term, count in misinfo_freq.most_common(top_n):
    pct = (count / len(misinfo_terms)) * 100
    print(f"  {term:20} | {count:6,} ({pct:5.2f}%)")

In [ ]:
# Identify distinctive terms (TF-IDF-like analysis)
# Terms that appear much more in one class than the other

control_total = len(control_terms)
misinfo_total = len(misinfo_terms)

# Calculate relative frequencies
all_terms = set(control_terms) | set(misinfo_terms)
distinctiveness = {}

for term in all_terms:
    control_freq_pct = (control_freq[term] / control_total) if control_total > 0 else 0
    misinfo_freq_pct = (misinfo_freq[term] / misinfo_total) if misinfo_total > 0 else 0
    
    if control_freq_pct > 0 and misinfo_freq_pct > 0:
        # Log ratio (distinctive if >> 1 or << 1)
        ratio = misinfo_freq_pct / control_freq_pct if control_freq_pct > 0 else float('inf')
        distinctiveness[term] = ratio

# Sort by distinctiveness (most associated with misinformation vs control)
sorted_distinctive = sorted(distinctiveness.items(), key=lambda x: x[1])

print("\n" + "="*70)
print("DISTINCTIVE TERMS")
print("="*70)

print(f"\n{top_n} Terms MORE associated with CONTROL:")
for term, ratio in sorted_distinctive[-top_n:]:
    print(f"  {term:20} | Ratio: {ratio:8.2f}x")

print(f"\n{top_n} Terms MORE associated with MISINFORMATION:")
for term, ratio in sorted_distinctive[:top_n]:
    print(f"  {term:20} | Ratio: {ratio:8.2f}x")

In [ ]:
# Visualize distinctive terms
distinctive_misinfo = sorted_distinctive[:10]
distinctive_control = sorted_distinctive[-10:]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Misinformation-associated terms
ax = axes[0]
terms_m = [t[0] for t in distinctive_misinfo]
ratios_m = [t[1] for t in distinctive_misinfo]
ax.barh(terms_m, ratios_m, color='#e74c3c')
ax.set_xlabel('Frequency Ratio (Misinfo / Control)', fontsize=11)
ax.set_title('Terms More Frequent in Misinformation', fontsize=12, fontweight='bold')
ax.invert_yaxis()

# Control-associated terms
ax = axes[1]
terms_c = [t[0] for t in distinctive_control]
ratios_c = [1 / t[1] for t in distinctive_control]  # Invert for clarity
ax.barh(terms_c, ratios_c, color='#2ecc71')
ax.set_xlabel('Frequency Ratio (Control / Misinfo)', fontsize=11)
ax.set_title('Terms More Frequent in Control (Science)', fontsize=12, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 5. N-gram Analysis

In [ ]:
# Extract bigrams (2-word phrases)
def extract_ngrams(text, n=2, remove_stopwords=True):
    """Extract n-grams from text."""
    if not isinstance(text, str):
        return []
    
    tokens = extract_terms(text, remove_stopwords=remove_stopwords, min_length=3)
    ngrams = [' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    
    return ngrams

# Extract bigrams
control_bigrams = []
misinfo_bigrams = []

for idx, row in df_pd.iterrows():
    bigrams = extract_ngrams(row['body'], n=2)
    if row['label'] == 0:
        control_bigrams.extend(bigrams)
    else:
        misinfo_bigrams.extend(bigrams)

print(f"✓ Extracted {len(control_bigrams):,} control bigrams")
print(f"✓ Extracted {len(misinfo_bigrams):,} misinformation bigrams")

In [ ]:
# Top bigrams by class
control_bigram_freq = Counter(control_bigrams)
misinfo_bigram_freq = Counter(misinfo_bigrams)

top_n = 12

print("\n" + "="*70)
print("TOP BIGRAMS BY CLASS")
print("="*70)

print(f"\nControl (Science) - Top {top_n} bigrams:")
for bigram, count in control_bigram_freq.most_common(top_n):
    pct = (count / len(control_bigrams)) * 100
    print(f"  {bigram:35} | {count:6,}")

print(f"\nMisinformation - Top {top_n} bigrams:")
for bigram, count in misinfo_bigram_freq.most_common(top_n):
    pct = (count / len(misinfo_bigrams)) * 100
    print(f"  {bigram:35} | {count:6,}")

In [ ]:
# Visualize top bigrams
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_control_bigrams = control_bigram_freq.most_common(10)
top_misinfo_bigrams = misinfo_bigram_freq.most_common(10)

# Control bigrams
ax = axes[0]
bigrams_c = [b[0] for b in top_control_bigrams]
counts_c = [b[1] for b in top_control_bigrams]
ax.barh(bigrams_c, counts_c, color='#2ecc71')
ax.set_xlabel('Frequency', fontsize=11)
ax.set_title('Top 10 Bigrams - Control (Science)', fontsize=12, fontweight='bold')
ax.invert_yaxis()

# Misinformation bigrams
ax = axes[1]
bigrams_m = [b[0] for b in top_misinfo_bigrams]
counts_m = [b[1] for b in top_misinfo_bigrams]
ax.barh(bigrams_m, counts_m, color='#e74c3c')
ax.set_xlabel('Frequency', fontsize=11)
ax.set_title('Top 10 Bigrams - Misinformation', fontsize=12, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Author Participation Patterns

In [ ]:
# Analyze author participation patterns
print("\n" + "="*70)
print("AUTHOR PARTICIPATION ANALYSIS")
print("="*70)

author_stats = df_ling.groupBy('author', 'label').agg(
    count('*').alias('n_comments'),
    avg('score').alias('avg_score'),
    avg('word_count').alias('avg_word_count')
).toPandas()

# Summary by class
class_author_stats = author_stats.groupby('label').agg({
    'n_comments': ['min', 'max', 'mean', 'median', 'count'],
    'avg_score': ['mean'],
    'avg_word_count': ['mean']
}).round(2)

print("\nAuthor participation by class:")
for label in [0, 1]:
    label_name = 'Control' if label == 0 else 'Misinformation'
    label_data = author_stats[author_stats['label'] == label]
    
    print(f"\n{label_name}:")
    print(f"  Unique authors:     {label_data['author'].nunique():,}")
    print(f"  Avg comments/author: {label_data['n_comments'].mean():.1f}")
    print(f"  Max comments/author: {label_data['n_comments'].max():,}")
    print(f"  Median score:        {label_data['avg_score'].median():.1f}")
    print(f"  Avg word count:      {label_data['avg_word_count'].mean():.1f}")

In [ ]:
# Visualize author participation distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comments per author
ax = axes[0]
for label, color, name in [(0, '#2ecc71', 'Control'), (1, '#e74c3c', 'Misinformation')]:
    data = author_stats[author_stats['label'] == label]['n_comments']
    ax.hist(data, bins=50, alpha=0.6, label=name, color=color)

ax.set_xlabel('Comments per Author', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Author Participation Distribution', fontsize=12, fontweight='bold')
ax.set_xlim(0, 50)  # Focus on main range
ax.legend()

# Score distribution
ax = axes[1]
author_stats.boxplot(column='avg_score', by='label', ax=ax)
ax.set_title('Average Score by Author Class', fontsize=12, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Average Score')
ax.set_xticklabels(['Control', 'Misinformation'])
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.show()

## 7. Sentiment & Tone Analysis (Proxy)

In [ ]:
# Use VADER sentiment analysis as a proxy for tone
sia = SentimentIntensityAnalyzer()

sentiments = []
for idx, row in df_pd.iterrows():
    scores = sia.polarity_scores(row['body'])
    sentiments.append({
        'compound': scores['compound'],  # -1 (negative) to +1 (positive)
        'positive': scores['pos'],
        'negative': scores['neg'],
        'neutral': scores['neu'],
        'label': row['label']
    })

sentiment_df = pd.DataFrame(sentiments)

print(f"✓ Computed sentiment for {len(sentiment_df)} comments")

In [ ]:
# Analyze sentiment by class
print("\n" + "="*70)
print("SENTIMENT ANALYSIS BY CLASS")
print("="*70)

sentiment_stats = sentiment_df.groupby('label').agg({
    'compound': ['mean', 'median', 'std'],
    'positive': 'mean',
    'negative': 'mean',
    'neutral': 'mean'
}).round(3)

for label in [0, 1]:
    label_name = 'Control' if label == 0 else 'Misinformation'
    label_data = sentiment_df[sentiment_df['label'] == label]
    
    print(f"\n{label_name}:")
    print(f"  Mean compound sentiment:  {label_data['compound'].mean():+.3f}")
    print(f"  Median compound:          {label_data['compound'].median():+.3f}")
    print(f"  Mean positive:            {label_data['positive'].mean():.3f}")
    print(f"  Mean negative:            {label_data['negative'].mean():.3f}")

In [ ]:
# Visualize sentiment analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Compound sentiment
ax = axes[0, 0]
for label, color, name in [(0, '#2ecc71', 'Control'), (1, '#e74c3c', 'Misinformation')]:
    data = sentiment_df[sentiment_df['label'] == label]['compound']
    ax.hist(data, bins=50, alpha=0.6, label=name, color=color)

ax.set_xlabel('Compound Sentiment Score', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Compound Sentiment Distribution', fontsize=12, fontweight='bold')
ax.legend()

# Sentiment components
ax = axes[0, 1]
components = ['positive', 'negative', 'neutral']
x = np.arange(len(components))
width = 0.35

control_means = [sentiment_df[sentiment_df['label'] == 0][c].mean() for c in components]
misinfo_means = [sentiment_df[sentiment_df['label'] == 1][c].mean() for c in components]

ax.bar(x - width/2, control_means, width, label='Control', color='#2ecc71')
ax.bar(x + width/2, misinfo_means, width, label='Misinformation', color='#e74c3c')

ax.set_xlabel('Sentiment Component', fontsize=11)
ax.set_ylabel('Average Score', fontsize=11)
ax.set_title('Sentiment Components by Class', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(components)
ax.legend()

# Box plots
ax = axes[1, 0]
sentiment_df.boxplot(column='compound', by='label', ax=ax)
ax.set_title('Compound Sentiment Box Plot', fontsize=12, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Compound Score')
ax.set_xticklabels(['Control', 'Misinformation'])
plt.suptitle('')

# Negative vs Positive
ax = axes[1, 1]
for label, color, name in [(0, '#2ecc71', 'Control'), (1, '#e74c3c', 'Misinformation')]:
    data = sentiment_df[sentiment_df['label'] == label]
    ax.scatter(data['positive'], data['negative'], alpha=0.3, label=name, color=color, s=20)

ax.set_xlabel('Positive Score', fontsize=11)
ax.set_ylabel('Negative Score', fontsize=11)
ax.set_title('Positive vs Negative Sentiment', fontsize=12, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Feature Correlation with Class

In [ ]:
# Combine all features for correlation analysis
combined_features = df_ling.select(
    'label', 'word_count', 'sentence_count', 'avg_word_length',
    'capitalization_ratio', 'digit_count', 'question_marks',
    'exclamation_marks', 'url_count', 'score'
).toPandas()

# Add sentiment features
combined_features = combined_features.reset_index(drop=True)
sentiment_df_reset = sentiment_df.reset_index(drop=True)
combined_features = pd.concat(
    [combined_features, sentiment_df_reset[['compound', 'positive', 'negative']]], 
    axis=1
)

# Compute correlation with label
correlations = combined_features.corr()['label'].drop('label').sort_values()

print("\n" + "="*70)
print("FEATURE CORRELATION WITH CLASS")
print("="*70)

print("\nFeatures most associated with CONTROL (negative correlation):")
for feature, corr in correlations.head(8).items():
    print(f"  {feature:25} | {corr:+.4f}")

print("\nFeatures most associated with MISINFORMATION (positive correlation):")
for feature, corr in correlations.tail(8).items():
    print(f"  {feature:25} | {corr:+.4f}")

In [ ]:
# Visualize correlations
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in correlations.values]
ax.barh(range(len(correlations)), correlations.values, color=colors)

ax.set_yticks(range(len(correlations)))
ax.set_yticklabels(correlations.index)
ax.set_xlabel('Correlation with Misinformation Label', fontsize=11)
ax.set_title('Feature Importance: Correlation with Class', fontsize=12, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Predicts Misinformation'),
    Patch(facecolor='#2ecc71', label='Predicts Control')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 9. Summary of Findings

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS FROM EXPLORATORY ANALYSIS")
print("="*70)

print("""
### Linguistic Markers
- Misinformation comments have distinctive linguistic profiles
- Key differences in capitalization, punctuation, and text complexity
- Specific terms and phrases are more frequent in each class

### Vocabulary Patterns
- Misinformation communities use specialized terminology
- Control communities focus on scientific/technical language
- Bigrams show domain-specific discourse patterns

### Author Behavior
- Participation patterns differ between communities
- Comment engagement (scores) varies by community
- Author vocabulary richness differs by class

### Sentiment & Tone
- Compound sentiment differs between classes
- Emotional content (positive/negative) varies
- Neutral language more prevalent in control communities

### Predictive Features
- Strong correlations exist between text features and class
- Multiple features can serve as predictors
- Combination of features should improve classification

### Next Steps
These features will be used in:
1. Baseline models (TF-IDF + Logistic Regression)
2. Deep learning models (BERT/DistilBERT fine-tuning)
3. Interpretability analysis of learned patterns
""")

print("="*70)

## Conclusion

This notebook demonstrated that **significant linguistic and behavioral differences exist between misinformation and control communities**. These differences can be captured through:

1. **Linguistic features** (word count, punctuation, capitalization)
2. **Vocabulary patterns** (distinctive terms, bigrams)
3. **Author behavior** (participation, engagement)
4. **Sentiment markers** (emotional tone, negativity)

**Next: Notebook 03** will build baseline machine learning models using these insights.